# Data Preparation
In brief, the data-preparation workflow begins by loading CIFTI, NIfTI, or GIFTI files and validating their fundamental properties, including data dimensionality, spatial representation, and cross-subject consistency. Based on these checks, an information dictionary is generated to summarize key dataset attributes and provide standardized metadata for subsequent processing steps. Downstream preprocessing procedures—such as space conversion, ROI selection, normalization, and data concatenation—are then performed systematically according to the information stored in this dictionary.

```{image} pic/data_prep/overall.png
:width: 800px
```

``fmriHA`` is a Python-based toolbox whose core computational routines operate primarily on data stored in **NumPy (`.npy`) format**. Therefore, before running hyperalignment, the original neuroimaging data need to be loaded, converted, and saved as `.npy` arrays that follow the expected input structure. This step ensures that subsequent functions can access subject-level response data efficiently and consistently.

In addition to the data format, ``fmriHA`` adopts a **BIDS-inspired naming and directory convention** to organize input and output files in a transparent and reproducible manner. File names are constructed using key–value pairs, such as `key1-value1_key2-value2_...`, so that relevant metadata are encoded directly in the file name. This convention improves readability, reduces ambiguity during automated file selection, and facilitates scalable processing across subjects, hemispheres, tasks, and analysis steps. A concrete example of this naming scheme is shown in the figure below.

```{image} pic/data_prep/dic_struct.png
:width: 800px
```

## Scripts Pipeline

Begin by importing the necessary packages.

In [ ]:
import joblib
import numpy as np
import neuroboros as nb
from pathlib import Path
from fmriha.preprocessing import prep

### (1) Input Validation

CIFTI and NIFTI files are validated with `prep.input_validation_nii`. GIFTI files are validated with `prep.input_validation_gii`. Both functions return a validation dictionary containing the subject list, session label, role label, input file metadata, grouped dimension information, missing files, and loading failures. This dictionary is the `input_info` argument used by the preprocessing functions below.

Parameters of `prep.input_validation_nii`:

| Parameter | Default | Meaning |
|---|---|---|
| `file_list` | Required | List of full CIFTI or NIFTI file paths. |
| `ses` | `"data"` | Session label written into output filenames as `ses-<label>`. |
| `bids_parse` | `True` | If `True`, parses BIDS-like filename entities such as `sub`, `run`, `task`, and `ses`. |
| `subject_key` | `"sub"` | Prefix used to locate subject IDs when `bids_parse=False`. |
| `check_dim` | `True` | If `True`, checks input image dimensions for consistency. |
| `group_by` | `None` | Filename entity used to group files before dimension checking, such as `"run"`. Use `None` to check all files together. |
| `role` | `"train"` | Dataset role written into output filenames as `set-<role>`. |
| `njobs` | `3` | Number of parallel workers used while reading headers and shapes. |
| `verbose` | `True` | If `True`, prints validation progress and detected shapes. |

Parameters of `prep.input_validation_gii`:

| Parameter | Default | Meaning |
|---|---|---|
| `file_list` | Required | List of hemisphere dictionaries. A two-hemisphere example is `{"l": left_file, "r": right_file}` for each subject/run. |
| `ses` | `"data"` | Session label written into output filenames as `ses-<label>`. |
| `bids_parse` | `True` | If `True`, parses BIDS-like filename entities from GIFTI filenames. |
| `subject_key` | `"sub"` | Prefix used to locate subject IDs when `bids_parse=False`. |
| `check_dim` | `True` | If `True`, checks dimensions within each hemisphere and group. |
| `group_by` | `"run"` | Filename entity used to group files before dimension checking. Use `None` to check all files together. |
| `role` | `"train"` | Dataset role written into output filenames as `set-<role>`. |
| `njobs` | `3` | Number of parallel workers used while reading headers and shapes. |
| `verbose` | `True` | If `True`, prints validation progress and detected shapes. |

Parameters of `prep.write_json_input_validate`:

| Parameter | Default | Meaning |
|---|---|---|
| `step_info` | Required | Validation dictionary returned by `prep.input_validation_nii` or `prep.input_validation_gii`. |
| `json_path` | Required | Path to the JSON file used to store validation records, usually `work_dir / "logs" / "prep_log.json"`. |
| `overwrite` | `False` | If `True`, starts a fresh validation log. If `False`, appends the validation record. |

The following cells define example CIFTI paths and subject IDs from the **Raiders** dataset, where run-01 uses BIDS-like filenames such as `sub-rid000005_ses-raiders_run-01_TS.dtseries.nii`, while run-02 uses custom filenames such as `xSfrNGRd_rid000005_raiders_run-02_TS.dtseries.nii`.

In [ ]:
data_dir = Path("/path/to/raw_data")
work_dir = Path("/path/to/work_dir")
json_path = work_dir / "logs" / "prep_log.json"
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

For BIDS-like filenames, set `bids_parse=True`.

In [ ]:
data_list_run1 = [
    data_dir / f"{sub}_ses-raiders_run-01_TS.dtseries.nii"
    for sub in sub_list
]

data_info_run1 = prep.input_validation_nii(
    file_list=data_list_run1,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info_run1, json_path, overwrite=True)

For custom filenames, set `bids_parse=False` and provide the prefix that marks the subject identifier through `subject_key`. In the Raiders example, run-02 uses filenames beginning with `xSfrNGRd_<subject>`, so `subject_key="xSfrNGRd"` tells fMRI-HA how to recover the subject ID.

In [ ]:
data_list_run2 = [
    data_dir / f"xSfrNGRd_{sub.split('-')[1]}_raiders_run-02_TS.dtseries.nii"
    for sub in sub_list
]

data_info_run2 = prep.input_validation_nii(
    file_list=data_list_run2,
    ses="raiders",
    bids_parse=False,
    subject_key="xSfrNGRd",
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info_run2, json_path, overwrite=False)

### (2) Data Preprocessing

After validation, the preprocessing functions write prepared arrays under `work_dir / sub-<subject> / resample / response / <structure>`. Surface outputs use `hemi-L` and `hemi-R`; CIFTI-derived subcortical outputs use folders such as `volume-subcortical-L`; NIFTI ROI outputs use `volume-<roi_name>`.

`normalize` accepts `"zscore"`, `"demean"`, or `None`. When `normalize=None`, the data values are saved in raw form and the filename uses `desc-bold-raw`.

For the CIFTI examples below, `data_info_bids` contains the validated Raiders run-01 files and `data_info` contains the validated Raiders run-02 files. The preparation calls are written once per run so that both runs are saved as prepared arrays and can be selected later with `run=["01", "02"]`.

**CIFTI Cortical Data**

CIFTI cortical data are extracted from `.dtseries.nii` or `.dscalar.nii` files and saved as separate left and right cortical matrices. `prep.prep_surf` loads the cortical data, restores medial-wall vertices for CIFTI inputs when needed, applies surface mapping when requested, applies the target cortical or ROI mask, normalizes the matrix, and saves `.npy` files.

Parameters of `prep.prep_surf`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root output directory. Prepared files, logs, and masks are written inside this directory. |
| `input_info` | Required | Validation dictionary returned by `prep.input_validation_nii` or `prep.input_validation_gii`. |
| `do_resample` | `True` | If `True`, applies a surface mapper before masking. Use `False` when source and target spaces already match. |
| `res_param` | `None` | Dictionary describing the surface mapping and output resolution label. Required when `do_resample=True`. |
| `do_medial_rm` | `True` | If `True`, applies the mask specified by `medial_param`. This can remove the medial wall or extract a surface ROI. |
| `medial_param` | `None` | Dictionary describing the target-surface mask. Required when `do_medial_rm=True`. |
| `normalize` | `"zscore"` | Normalization method applied along the time axis for each vertex. Use `"zscore"`, `"demean"`, or `None`. |
| `dtype` | `np.float32` | Numeric dtype used while preparing and saving arrays. |
| `njobs` | `3` | Number of parallel workers used for file-wise preparation. |
| `verbose` | `5` | Joblib verbosity level. |
| `json_path` | `"prep_log.json"` | JSON path used to record preprocessing settings. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |

Fields commonly used in `res_param` for surface data:

| Parameter | Default | Meaning |
|---|---|---|
| `source` | Required when resampling without provided mapper data | Source surface space of the input data, such as `fslr-ico57`. |
| `source_mask` | `None` | Source-space mask used by the mapper. Provide a mask when source surface data have already had vertices removed. |
| `target` | Required when resampling without provided mapper data | Target analysis space, such as `onavg-ico32`. |
| `target_mask` | `None` | Target-space mask used by the mapper. Set this to `True` when the `neuroboros` mapper should project directly to the cortical target vertices; otherwise, the final cortical or ROI selection is usually controlled by `medial_param`. |
| `data` | `None` | Mapper dictionary keyed by hemisphere (`"l"` and `"r"`). With `None`, the function loads the configured mapper from `neuroboros`. |
| `desc` | Recommended | Human-readable resampling description written to logs. |
| `res` | Recommended | Short resolution label written into output filenames as `res-<label>`. If omitted, `"mapped"` is used when resampling and `"raw"` is used when resampling is disabled. |

Fields commonly used in `medial_param` for surface data:

| Parameter | Default | Meaning |
|---|---|---|
| `space` | Required when loading a mask | Surface space used when loading a cortical mask through `neuroboros`. |
| `data` | `None` | Mask dictionary keyed by hemisphere (`"l"` and `"r"`). Vertices marked `True` are retained. |
| `roi_name` | Recommended | ROI label written into output filenames as `roi-<label>`. If omitted, `"whole"` is used. |
| `desc` | Recommended | Human-readable mask description written to logs. |

The example below uses the configured mapper and cortical mask for `fslr-ico57` to `onavg-ico32` preparation.


In [ ]:
res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": None,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": None,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

You can also provide your own mapper and target mask through `res_param["data"]` and `medial_param["data"]`.


In [ ]:
mapper_data = joblib.load("/path/to/mapper.pkl")
cortical_mask_data = joblib.load("/path/to/cortical_mask.pkl")

res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": mapper_data,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": cortical_mask_data,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

In [ ]:
# Raiders run-01
prep.prep_surf(
    work_dir,
    input_info=data_info_run1,
    do_resample=True,
    res_param=res_param,
    do_medial_rm=True,
    medial_param=medial_param,
    normalize="zscore",
    dtype=np.float32,
    njobs=3,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="cifti_run01",
)

# Raiders run-02
prep.prep_surf(
    work_dir,
    input_info=data_info_run2,
    do_resample=True,
    res_param=res_param,
    do_medial_rm=True,
    medial_param=medial_param,
    normalize="zscore",
    dtype=np.float32,
    njobs=3,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="cifti_run02",
)


**CIFTI Subcortical Data**

CIFTI subcortical data are extracted from the CIFTI BrainModels axis and saved as volume-derived `time x voxel` matrices. The standard GUI-compatible outputs are `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`.

Parameters of `prep.prep_volume`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root output directory. Prepared files, logs, and masks are written inside this directory. |
| `input_info` | Required | Validation dictionary returned by `prep.input_validation_nii`. |
| `cifti_region` | `"whole"` | CIFTI subcortical selector passed to the CIFTI loader. Common values are `"L"`, `"R"`, `"BRAIN_STEM"`, `"whole"`, a full CIFTI structure name, or a list of structures. |
| `roi_name` | `"cortical_cift"` | ROI label used in output folder names and filenames. |
| `do_resample` | `True` | If `True`, resamples the volume before masking. |
| `res_param` | `None` | Volume resampling dictionary. Required when `do_resample=True`. |
| `mask` | `None` | External 3D NIFTI mask for NIFTI input. For CIFTI input, voxel selection is derived from `cifti_region` and the CIFTI BrainModels axis. |
| `save_resample_mask` | `True` | If `True`, saves a representative 3D mask in `work_dir / "masks"`. |
| `normalize` | `"zscore"` | Normalization method applied along the time axis for each voxel. Use `"zscore"`, `"demean"`, or `None`. |
| `dtype` | `np.float32` | Numeric dtype used while preparing and saving arrays. |
| `njobs` | `3` | Number of parallel workers used for file-wise preparation. |
| `verbose` | `5` | Joblib verbosity level. |
| `json_path` | `"prep_log.json"` | JSON path used to record preprocessing settings. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |

Fields commonly used in `res_param` for volume data:

| Parameter | Default | Meaning |
|---|---|---|
| `ref` | Required for `mode="ref"` | Reference NIFTI image whose grid should be matched. |
| `interpolation` | Required when resampling | Interpolation method passed to the volume resampling helper. Common values are `"nearest"`, `"linear"`, `"continuous"`, `"spline"`, and `"multilinear"`. |
| `vol_size` | Required for `mode="vox"` | Target isotropic voxel size. The key is also expected in reference-based workflows, where it may be set to `None` or recorded as metadata. |
| `mode` | Required when resampling | `"ref"` matches the grid of `ref`; `"vox"` creates a grid from `vol_size`. |
| `res` | `"nm"` when omitted | Short resolution label written into output filenames as `res-<label>`. |

For CIFTI subcortical preparation in the native CIFTI-derived grid, use `do_resample=False`, `res_param=None`, and `mask=None`.


In [ ]:
subcortical_regions = ["L", "R", "BRAIN_STEM"]

# Raiders run-01
for structure_name in subcortical_regions:
    prep.prep_volume(
        work_dir,
        input_info=data_info_run1,
        cifti_region=structure_name,
        roi_name=f"subcortical-{structure_name}",
        do_resample=False,
        res_param=None,
        mask=None,
        save_resample_mask=True,
        normalize="zscore",
        dtype=np.float32,
        njobs=2,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num=f"cifti_subcortical-{structure_name}",
    )

# Raiders run-02
for structure_name in subcortical_regions:
    prep.prep_volume(
        work_dir,
        input_info=data_info_run2,
        cifti_region=structure_name,
        roi_name=f"subcortical-{structure_name}",
        do_resample=False,
        res_param=None,
        mask=None,
        save_resample_mask=True,
        normalize="zscore",
        dtype=np.float32,
        njobs=2,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num=f"cifti_subcortical-{structure_name}",
    )

**GIFTI Cortical Data**

GIFTI preparation is surface-only. Each entry in `file_list` is a dictionary whose keys identify hemispheres. Use `{"l": left_gifti, "r": right_gifti}` when both hemispheres are present. After `prep.input_validation_gii`, use `prep.prep_surf` with the same surface parameter structure shown above.

Prepared GIFTI outputs are saved in `hemi-L` and `hemi-R` folders, matching the structure names used by downstream hyperalignment functions. The example below assumes fMRIPrep-style fsaverage surface files represented as `fsavg-ico128` in the mapper. Because `target_mask=True` maps directly to cortical target vertices, the separate medial-wall removal step is disabled.


In [ ]:
res_param = {
    "source": "fsavg-ico128",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": True,
    "data": None,
    "desc": "fsavg-ico128 whole to onavg-ico32 cortical",
    "res": "onavg32",
}

for run in ['01', '02']:
    train_list = []

    for sub in sub_list:
        train_list.append(
            {
                "l": data_dir / f"{sub}_task-raiders_acq-8ch336vol_run-{run}_hemi-L_space-fsaverage_bold.func.gii",
                "r": data_dir / f"{sub}_task-raiders_acq-8ch336vol_run-{run}_hemi-R_space-fsaverage_bold.func.gii",
            }
        )

    gifti_info = prep.input_validation_gii(
        file_list=train_list,
        ses="raiders",
        bids_parse=True,
        subject_key=None,
        check_dim=True,
        group_by="run",
        role="train",
        njobs=2,
        verbose=True,
    )

    prep.write_json_input_validate(gifti_info, json_path, overwrite=False)

    prep.prep_surf(
        work_dir,
        input_info=gifti_info,
        do_resample=True,
        res_param=res_param,
        do_medial_rm=False,
        medial_param=None,
        normalize="zscore",
        dtype=np.float32,
        njobs=3,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num="gifti_demo",
    )

**NIFTI Volume Data**

NIFTI volume preparation uses `prep.input_validation_nii` followed by `prep.prep_volume`. Features are selected with a 3D NIFTI mask when `mask` is provided. The `roi_name` value becomes the output structure folder, for example `volume-cortical` or `volume-mPFC`.

Set `cifti_region=None` for NIFTI input. Volume resampling is controlled by `res_param`, and two modes are commonly used:

- `mode="ref"`: resample each input image to match a reference NIFTI image specified by `ref`. This is useful when the prepared data should share the same grid as an atlas or ROI mask.
- `mode="vox"`: resample each input image according to a target isotropic voxel size specified by `vol_size`. This is useful when you want to define the output resolution directly.

Choose one `res_param` definition for a given preprocessing run. The example below uses the reference-image mode by default and also shows the voxel-size mode as an alternative.


In [ ]:
# Option 1: resample to a reference NIFTI image.
# The `res` field is used when constructing output filenames.
res_param = {
    "ref": "/path/to/HCPex_2mm_cortical_mask.nii",
    "interpolation": "nearest",
    "vol_size": 2,
    "mode": "ref",
    "res": "2mm",
}

# Option 2: resample according to a target voxel size.
res_param = {
    "ref": None,
    "interpolation": "nearest",
    "vol_size": 3,
    "mode": "vox",
    "res": "3mm",
}

Next, define a volumetric ROI mask. The mask is applied after volume resampling. If the mask does not match the resampled data grid, it will be resampled automatically and the relevant information will be reported.

In [ ]:
mask = "/path/to/HCPex_2mm_cortical_mask.nii"

In [ ]:
for run in ['01', '02']:
    nifti_list = [
        data_dir / f"{sub}_task-raiders_acq-8ch336vol_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
        for sub in sub_list
    ]

    nifti_info = prep.input_validation_nii(
        file_list=nifti_list,
        ses="raiders",
        bids_parse=True,
        subject_key=None,
        check_dim=True,
        group_by="run",
        role="train",
        njobs=2,
        verbose=True,
    )

    prep.write_json_input_validate(nifti_info, json_path, overwrite=False)

    prep.prep_volume(
        work_dir,
        input_info=nifti_info,
        cifti_region=None,
        roi_name="cortical",
        do_resample=True,
        res_param=res_param,
        mask=mask,
        save_resample_mask=True,
        normalize="zscore",
        dtype=np.float32,
        njobs=2,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num="nifti_demo",
    )


**Concatenating Prepared Files**

After preparing individual runs or modules, use `prep.combine_file` to concatenate matching `.npy` files along the first axis, usually the time dimension. Concatenation is performed separately for each subject and requested structure.

Parameters of `prep.combine_file`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root directory containing subject folders. |
| `sub_list` | Required | Subject folder names to process, such as `"sub-rid000005"`. |
| `step` | Required | Processing step folder to search, usually `"resample"`. |
| `modality` | Required | Modality folder under `step`, commonly `"response"` or `"connectivity"`. `"mixed"` is not supported. |
| `structures` | Required | Structure folder or list of folders to concatenate, such as `"hemi-L"` or `"volume-subcortical-L"`. |
| `ses_name` | Required | Session label assigned to the concatenated output filename. |
| `njobs` | Required | Number of parallel workers across subject-structure jobs. |
| `method` | `"zscore"` | Optional normalization applied after concatenation. Use `"zscore"`, `"demean"`, or `None`. |
| `json_path` | `"prep_log.json"` | JSON path used to record the concatenation step. |
| `dtype` | `"np.float32"` | Output numeric dtype used when loading and post-processing arrays. Both NumPy dtypes and strings such as `"float32"` are accepted. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `rename_fields` | `None` | BIDS-style filename fields to replace in the output filename. If `None`, only `ses` is replaced with `ses_name`. |
| `**file_filter` | Empty | BIDS-like filename filters used to select files, such as `run=["01", "02"]`, `set="train"`, or `roi="cortical"`. |

`file_filter` is matched against filename entities. Add filters such as `space`, `roi`, `res`, `desc`, or `set` when a folder contains multiple candidate files. In the following example, we concatenate data from two runs.


In [ ]:
file_filter = {
    "run": ["01", "02"],
}

prep.combine_file(
    work_dir,
    sub_list,
    step="resample",
    modality="response",
    structures=[
        "hemi-L",
        "hemi-R",
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    ses_name="comb12",
    njobs=2,
    method=None,
    json_path=json_path,
    dtype=np.float32,
    overwrite=False,
    log_num="combine_runs",
    **file_filter,
)

## GUI Pipeline

Launch the graphical interface with `gui()`.

In [ ]:
from fmriha import gui
gui()

```{image} pic/data_prep/gui.png
:width: 800px
```

At the very top of the main page is the output directory, **Work Dir**. You can either click the **···** button to open the file manager and select the corresponding directory, or directly enter your output path in the input box.

```{image} pic/data_prep/gui_workdir.png
:width: 800px
```

Next, click the **n_jobs Parameters** button and specify the number of parallel CPU cores for the Data Preparation step.

```{image} pic/data_prep/gui_njobs.png
:width: 800px
```

If you want to process Surface data, please click the **Space and Searchlight Configuration** button and select **Space** in the **Surface** module. We recommend choosing **onavg-ico32** here.

If you want to apply a custom mask manually, specify it in Custom Mask. Only .pkl files are supported. The file should contain a dictionary in the following format:

In [ ]:
{
    'l': left_mask,
    'r': right_mask
}

```{image} pic/data_prep/gui_space.png
:width: 800px
```

If you want to process Volume data in NIFTI, please provide the **NIFTI Mask** and the corresponding **ROI Name**.

```{image} pic/data_prep/gui_nifti_mask.png
:width: 800px
```

If you want to process volume data in CIFTI, no additional configuration is required in **Space and Searchlight Configuration**.

Next, click the **Data Preparation** button on the main page. On this page, you can provide different datasets for different modules. You can decide which data to use for the Template, calculating the Transformation Matrix, and applying the Transformation Matrix (Alignment).

If you want to provide data for template computation, you need to check the box in front of Template. The same applies to Transformation Data and Aligned Data.

```{image} pic/data_prep/gui_datachoose.png
:width: 800px
```

The following operations will use Template data as an example.

We support three types of data: **CIFTI**, **GIFTI**, and **NIFTI**, which can be selected under **Data Type**.

If you choose **CIFTI**, the available options in **Structure** include **hemi-L**, **hemi-R**, **subcortical-L**, **subcortical-R**, and **subcortical-BRAIN_STEM**. Among them, **hemi-L** and **hemi-R** correspond to cortical surface data, while **subcortical-L**, **subcortical-R**, and **subcortical-BRAIN_STEM** correspond to subcortical volumetric data.

Next, you can either click the **Add Files** button to open the file manager and select the desired files, or directly enter the corresponding file paths into the table.

If your **Structure** is set to the surface types **hemi-L** or **hemi-R**, you also need to specify the original space of the imported data in **Surface** (for example, **fslr-ico57** in the example below). In addition, if the surface portion of your CIFTI file has already had the medial wall removed, you need to provide the corresponding source-space mask in **Source Medial Wall Removed**.


```{image} pic/data_prep/gui_cifti.png
:width: 800px
```

Further down, you can choose the normalization method for the data (**raw** means no normalization will be applied). If your filenames follow the BIDS convention, simply check **BIDS File**.

```{image} pic/data_prep/gui_bids.png
:width: 800px
```

Otherwise, you need to fill in the **Subject Key** to specify the subject identifier (for example, **xSfrNGRd** in the example below).

```{image} pic/data_prep/gui_notbids.png
:width: 800px
```

If you select **GIFTI** as the **Data Type**, the workflow is very similar to that of **CIFTI**. The main difference is that you need to use **Add Files (hemi-L)** and **Add Files (hemi-R)** to load the left- and right-hemisphere files separately.

If you select **NIFTI** as the **Data Type**, choose **volume-ROI** in **Structure** and fill in the corresponding information in the **Volume (NIFTI)** module. **Mask** should point to the 3D NIFTI mask used to select voxels, and **NIFTI ROI Name** defines the `roi_name` used in output folder names and filenames.


```{image} pic/data_prep/gui_nifti.png
:width: 800px
```

If you want to use files from multiple runs, you can click the "**+**" button at the lower right corner of each module first.

```{image} pic/data_prep/gui_addmodule.png
:width: 800px
```

Then check the **Combine Files** option at the bottom. The selected file filters are passed to `prep.combine_file`, so they should match the BIDS-style fields in the prepared filenames.

```{image} pic/data_prep/gui_combine.png
:width: 800px
```

For Transformation Data and Aligned Data, if you want to directly reuse the files from the previous modules, you can check the corresponding **Same as** options. Here, **Same as Tmpl. Data** means using the same data as the Template module, while **Same as T Data** means using the same data as the Transformation Matrix module. After clicking a **Same as** option, you will no longer be able to modify the contents within the current module. Any changes must be made in the source module that the settings were copied from.

In addition, the **Reset All Settings** button on the **Data Preparation** page only clears the settings within the module where the button is located; it will not affect the settings of other modules. The **Reset All Settings button** on the **main page**, however, applies globally and resets settings across all modules.

After completing all the configurations above, you can directly click the **RUN!** button on the main page to execute Data Preparation. Alternatively, you can first finish configuring all RHA/CHA settings and then run the entire pipeline together using the **RUN!** button.